# Music Genre Classification with Google LEAF
Train a CNN classifier using PyTorch and Google's Learnable Audio Frontend (LEAF) on the GTZAN dataset.
Make sure your runtime is set to **T4 GPU**.

In [ ]:
!pip install -qU torch torchaudio soundfile scikit-learn
!git clone https://github.com/SarthakYadav/leaf-pytorch.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 41.4 MB/s eta 0:00:00
Cloning into 'leaf-pytorch'...
remote: Enumerating objects: 311, done.
remote: Counting objects: 100% (311/311), done.
remote: Compressing objects: 100% (125/125), done.
remote: Total 311 (delta 184), reused 301 (delta 177), pack-reused 0 (from 0)
Receiving objects: 100% (311/311), 93.18 KiB | 2.03 MiB/s, done.
Resolving deltas: 100% (184/184), done.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import soundfile as sf
import os
import sys
import torchaudio
import urllib.request
import tarfile

sys.path.append(os.path.join(os.getcwd(), 'leaf-pytorch'))
from leaf_pytorch.frontend import Leaf

In [ ]:
!export KAGGLE_API_TOKEN=YOUR_KAGGLE_API_TOKEN

In [ ]:
import kagglehub

path = kagglehub.dataset_download("andradaolteanu/gtzan-dataset-music-genre-classification")

print("Path to dataset files:", path)

100%|██████████| 1.21G/1.21G [00:13<00:00, 99.7MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/andradaolteanu/gtzan-dataset-music-genre-classification/versions/1


In [ ]:
import shutil
import os
from pathlib import Path

# Source path from kagglehub
source_path = path
# Destination path in Colab's content directory
dest_path = '/content/gtzan'

if not Path(dest_path).exists():
    print(f"Moving dataset from {source_path} to {dest_path}...")
    shutil.copytree(source_path, dest_path)
    print("Move complete! You should now see 'gtzan_dataset' in your file explorer.")
else:
    print("Dataset already exists in /content/")

Moving dataset from /root/.cache/kagglehub/datasets/andradaolteanu/gtzan-dataset-music-genre-classification/versions/1 to /content/gtzan...
Move complete! You should now see 'gtzan_dataset' in your file explorer.


In [ ]:
import zipfile
import os

def extract_kaggle_archive(zip_path="archive.zip", extract_to="."):
    """Extracts the Kaggle archive.zip into the current directory."""
    target_dir = os.path.join(extract_to, "Data", "genres_original")

    if os.path.exists(target_dir) and len(os.listdir(target_dir)) > 0:
        print(f"Dataset already extracted at {target_dir}")
        return target_dir

    if not os.path.exists(zip_path):
        raise FileNotFoundError(f"Could not find {zip_path}. Please upload it to Colab.")

    print(f"Extracting {zip_path} to {extract_to}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)

    print("Extraction complete!")
    return target_dir

class GTZANDataset(Dataset):
    def __init__(self, root="gtzan/Data/genres_original", sample_rate=16000, duration=5.0):
        self.root = root
        self.target_sample_rate = sample_rate
        self.max_length = int(sample_rate * duration)

        self.genres = [
            "blues", "classical", "country", "disco", "hiphop",
            "jazz", "metal", "pop", "reggae", "rock"
        ]
        self.genre_to_idx = {g: i for i, g in enumerate(self.genres)}
        self.resamplers = {}

        self.files = []
        for genre in self.genres:
            genre_dir = os.path.join(self.root, genre)
            if not os.path.exists(genre_dir):
                print(f"Warning: Directory not found - {genre_dir}")
                continue
            for f in os.listdir(genre_dir):
                if f.endswith('.wav') or f.endswith('.au'):
                    self.files.append((os.path.join(genre_dir, f), genre))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_path, label = self.files[idx]
        try:
            waveform, sr = torchaudio.load(file_path)
        except Exception:
            # Handle corrupted files (GTZAN has a known corrupted jazz file)
            waveform = torch.zeros(1, self.max_length)
            sr = self.target_sample_rate

        # Optimized resampling logic
        if sr != self.target_sample_rate:
            if sr not in self.resamplers:
                # Cache the resampler for this sample rate
                self.resamplers[sr] = torchaudio.transforms.Resample(
                    orig_freq=sr,
                    new_freq=self.target_sample_rate
                )
            # Use the cached resampler
            waveform = self.resamplers[sr](waveform)

        # Force mono
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        # Pad or truncate
        if waveform.shape[1] > self.max_length:
            waveform = waveform[:, :self.max_length]
        elif waveform.shape[1] < self.max_length:
            pad_amount = self.max_length - waveform.shape[1]
            waveform = F.pad(waveform, (0, pad_amount))

        label_idx = self.genre_to_idx[label]
        return waveform, label_idx


In [ ]:
# Verify files in each of the gtzan dataset by listing atlease one file from each genre
dataset = GTZANDataset()
[dataset.__dict__['files'][(i + 1) * 100 - 1] for i in range(10)]


[('gtzan/Data/genres_original/blues/blues.00086.wav', 'blues'),
 ('gtzan/Data/genres_original/classical/classical.00028.wav', 'classical'),
 ('gtzan/Data/genres_original/country/country.00057.wav', 'country'),
 ('gtzan/Data/genres_original/disco/disco.00004.wav', 'disco'),
 ('gtzan/Data/genres_original/hiphop/hiphop.00063.wav', 'hiphop'),
 ('gtzan/Data/genres_original/jazz/jazz.00074.wav', 'jazz'),
 ('gtzan/Data/genres_original/metal/metal.00028.wav', 'metal'),
 ('gtzan/Data/genres_original/pop/pop.00041.wav', 'pop'),
 ('gtzan/Data/genres_original/reggae/reggae.00094.wav', 'reggae'),
 ('gtzan/Data/genres_original/rock/rock.00048.wav', 'rock')]

In [ ]:
class AudioClassifier(nn.Module):
    def __init__(self, num_classes=10, sample_rate=16000, n_filters=40):
        super(AudioClassifier, self).__init__()

        # 1. Learnable Audio Frontend
        self.leaf = Leaf(
            sample_rate=sample_rate,
            n_filters=n_filters,
            init_min_freq=60.0,
            init_max_freq=7800.0
        )

        # 2. 2D CNN Backbone
        # Input shape from LEAF: (Batch, n_filters, Time)
        # We will unsqueeze it to: (Batch, 1, n_filters, Time) to treat it as a 2D image

        self.conv_block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Dropout2d(0.2)
        )

        self.conv_block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Dropout2d(0.3)
        )

        self.conv_block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Dropout2d(0.3)
        )

        self.conv_block4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)), # Pools (Freq, Time) down to 1x1
            nn.Dropout(0.4)
        )

        # 3. Classifier Head
        self.fc1 = nn.Linear(256, 128)
        self.dropout_fc = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        # 1. LEAF
        x = self.leaf(x)  # Shape: (B, 40, Time)

        # 2. Add channel dimension for 2D Convolutions
        x = x.unsqueeze(1) # Shape: (B, 1, 40, Time)

        # 3. 2D Convolutions
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = self.conv_block4(x)

        # 4. Flatten
        x = x.view(x.size(0), -1) # Shape: (B, 256)

        # 5. Fully Connected
        x = F.relu(self.fc1(x))
        x = self.dropout_fc(x)
        x = self.fc2(x)

        return x

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Path to the original genres datasest
data_path = Path(dest_path) / 'Data/genres_original'

# We already set data_path in the previous cell
print(f"Using data path: {data_path}")

full_dataset = GTZANDataset(root=data_path)
print(f"Total dataset size: {len(full_dataset)}")

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

model = AudioClassifier(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

scaler = torch.amp.GradScaler()

epochs = 100
print(f"Starting training for {epochs} epochs...")

for epoch in range(epochs):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            outputs = model(inputs)
            loss = criterion(outputs, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

    train_loss = total_loss / len(train_loader)
    train_acc = 100. * correct / total

    model.eval()
    val_loss = 0
    all_targets, all_preds = [], []

    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            all_targets.extend(targets.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())

    val_loss = val_loss / len(val_loader)
    val_acc = accuracy_score(all_targets, all_preds) * 100.
    val_f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)

    print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f}, Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f}, Acc: {val_acc:.2f}%, F1: {val_f1:.4f}")

Using device: cuda
Using data path: /content/gtzan/Data/genres_original
Total dataset size: 1000
torch.Size([40, 2])
Starting training for 30 epochs...
Epoch 01 | Train Loss: 2.2602, Acc: 14.75% | Val Loss: 2.3735, Acc: 8.50%, F1: 0.0157
Epoch 02 | Train Loss: 2.1430, Acc: 20.25% | Val Loss: 2.4321, Acc: 8.50%, F1: 0.0167
Epoch 03 | Train Loss: 2.0668, Acc: 21.88% | Val Loss: 2.1183, Acc: 17.50%, F1: 0.1013
Epoch 04 | Train Loss: 2.0352, Acc: 21.50% | Val Loss: 2.0043, Acc: 23.00%, F1: 0.1370
Epoch 05 | Train Loss: 2.0195, Acc: 22.50% | Val Loss: 2.0270, Acc: 21.50%, F1: 0.1287
Epoch 06 | Train Loss: 1.9959, Acc: 23.88% | Val Loss: 1.9940, Acc: 19.50%, F1: 0.1179
Epoch 07 | Train Loss: 1.9934, Acc: 23.38% | Val Loss: 1.9832, Acc: 23.50%, F1: 0.1650
Epoch 08 | Train Loss: 1.9338, Acc: 25.62% | Val Loss: 1.9565, Acc: 24.50%, F1: 0.1779
Epoch 09 | Train Loss: 1.9323, Acc: 25.00% | Val Loss: 1.9343, Acc: 20.00%, F1: 0.1418
Epoch 10 | Train Loss: 1.9041, Acc: 27.38% | Val Loss: 1.9533, Acc:

In [ ]:
# Save the trained model
model_name = "music_genre_classifier.pth"
torch.save(model.state_dict(), model_name)
print(f"Model saved to {model_name}")

Model saved to music_genre_classifier.pth
